In [19]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        #print(os.path.join(dirname, filename))
        pass

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [20]:
# --- Cell 1: Ultimate Dependency Fix for Kaggle (Must be the very first cell) ---

# 0. CRITICAL: Force Keras Backend
# This environment variable must be set BEFORE any Keras or TensorFlow imports.
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
# Set SM_FRAMEWORK as a secondary safety measure for segmentation-models
os.environ["SM_FRAMEWORK"] = "tf.keras"

# 1. Aggressive Uninstall (Remove lingering incompatible versions)
!pip uninstall -y keras tensorflow tf-keras-nightly tf-keras tensorflow-text tensorflow-decision-forests

# 2. Force Install Compatible Versions
print("Installing TensorFlow 2.15.0 and Keras 2.15.0...")
!pip install tensorflow==2.15.0 keras==2.15.0 --no-deps
!pip install Keras-Applications==1.0.8 --no-dependencies --no-cache-dir

# 3. Install specialized libraries
!pip install segmentation-models
!pip install tensorflow-addons 
!pip install keras-unet-collection 

# 4. Imports (Order matters here, as Keras must be loaded via TF)
import glob
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import tensorflow.keras.backend as K
import pandas as pd

# 5. CRITICAL Check and Fix
print(f"TensorFlow Version Check: {tf.__version__}")

# Clear V1 references if they persist (for keras-unet-collection)
try:
    del tf.compat.v1.image
    print("Cleaned up deprecated V1 references.")
except AttributeError:
    pass

# Import segmentation models (These should now resolve correctly)
try:
    # Use sm for the main module alias
    import segmentation_models as sm
    
    # Check for direct imports from the module
    from segmentation_models import Unet, AttentionUnet, UnetPlusPlus 
    
    print("Segmentation Models (sm) and core components imported successfully!")
except Exception as e:
    print(f"CRITICAL ERROR during segmentation_models import: {e}")
    
# Import Transformer-based models
try:
    from keras_unet_collection import models
    print("Keras-UNet-Collection imported successfully!")
except Exception as e:
    print(f"CRITICAL ERROR during keras_unet_collection import: {e}")

# --- Global Configuration (Keep these the same) ---
IMAGE_SIZE = (256, 256) 
INPUT_SHAPE = IMAGE_SIZE + (3,) 
BACKBONE = 'resnet34' 
BATCH_SIZE = 16 
N_CLASSES = 1

Found existing installation: keras 2.15.0
Uninstalling keras-2.15.0:
  Successfully uninstalled keras-2.15.0
Found existing installation: tensorflow 2.15.0
Uninstalling tensorflow-2.15.0:
  Successfully uninstalled tensorflow-2.15.0
Installing TensorFlow 2.15.0 and Keras 2.15.0...
  Using cached tensorflow-2.15.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.4 kB)
  Using cached keras-2.15.0-py3-none-any.whl.metadata (2.4 kB)
Using cached tensorflow-2.15.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (475.3 MB)
Using cached keras-2.15.0-py3-none-any.whl (1.7 MB)
TensorFlow Version Check: 2.18.0
CRITICAL ERROR during segmentation_models import: cannot import name 'AttentionUnet' from 'segmentation_models' (/usr/local/lib/python3.11/dist-packages/segmentation_models/__init__.py)
CRITICAL ERROR during keras_unet_collection import: cannot import name 'disable_resource_variables' from 'tensorflow.python.ops.variable_scope' (/usr/local/lib/python3.11/di

In [21]:
# --- Cell 2: Custom Metrics and Loss ---

SMOOTH = 1e-6 

# 1. DICE COEFFICIENT (Metric)
def dice_coef(y_true, y_pred):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + SMOOTH) / (K.sum(y_true_f) + K.sum(y_pred_f) + SMOOTH)

# 2. JACCARD INDEX (IoU) (Metric)
def iou_score(y_true, y_pred):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - intersection
    return (intersection + SMOOTH) / (union + SMOOTH)

# 3. SENSITIVITY (Recall) (Metric)
def sensitivity(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    return true_positives / (possible_positives + K.epsilon())

# 4. SPECIFICITY (Metric)
def specificity(y_true, y_pred):
    true_negatives = K.sum(K.round(K.clip((1 - y_true) * (1 - y_pred), 0, 1)))
    possible_negatives = K.sum(K.round(K.clip(1 - y_true, 0, 1)))
    return true_negatives / (possible_negatives + K.epsilon())

# HYBRID LOSS FUNCTION (Dice Loss + Binary Cross-Entropy)
def hybrid_loss(y_true, y_pred, alpha=0.5):
    """L_total = alpha * L_bce + (1 - alpha) * L_dice"""
    bce_loss = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice_loss = 1 - dice_coef(y_true, y_pred)
    return alpha * bce_loss + (1 - alpha) * dice_loss

METRICS = [iou_score, dice_coef, sensitivity, specificity, 'accuracy']

In [22]:
# --- Cell 3: Data Preparation and Generator ---

# --- Define Paths (Relative to your Kaggle Notebook directory) ---
FRAMES_DIR = 'archive/frames'
MASKS_DIR = 'archive/masks'

# --- Load and Sort Files ---
# Adjust the file extension if your frames are .jpg instead of .png
all_frames = sorted(glob.glob(os.path.join(FRAMES_DIR, '*.png')))
all_masks = sorted(glob.glob(os.path.join(MASKS_DIR, '*.png')))

if len(all_frames) == 0 or len(all_frames) != len(all_masks):
    print("FATAL ERROR: Data files not found or mismatched counts.")
else:
    print(f"Total paired samples found: {len(all_frames)}")

# --- Split Data ---
# 80% Train, 20% Validation
train_frames, val_frames, train_masks, val_masks = train_test_split(
    all_frames, all_masks, test_size=0.2, random_state=42
)
print(f"Train samples: {len(train_frames)}, Validation samples: {len(val_masks)}")


# --- Custom Data Generator (Keras Sequence) ---
class SegDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, frame_paths, mask_paths, batch_size, image_size, preprocess_input):
        self.frame_paths = frame_paths
        self.mask_paths = mask_paths
        self.batch_size = batch_size
        self.image_size = image_size
        self.preprocess_input = preprocess_input
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.frame_paths) / self.batch_size))

    def on_epoch_end(self):
        self.indexes = np.arange(len(self.frame_paths))
        np.random.shuffle(self.indexes)

    def __getitem__(self, index):
        indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]

        frame_batch_paths = [self.frame_paths[k] for k in indexes]
        mask_batch_paths = [self.mask_paths[k] for k in indexes]

        X, y = self.__data_generation(frame_batch_paths, mask_batch_paths)
        return X, y

    def __data_generation(self, frame_batch_paths, mask_batch_paths):
        X = np.empty((self.batch_size, *self.image_size, 3), dtype=np.float32)
        y = np.empty((self.batch_size, *self.image_size, N_CLASSES), dtype=np.float32)

        for i, (frame_path, mask_path) in enumerate(zip(frame_batch_paths, mask_batch_paths)):
            # Load, convert to array, and preprocess image
            img_arr = img_to_array(load_img(frame_path, target_size=self.image_size, color_mode='rgb'))
            X[i,] = self.preprocess_input(img_arr) 

            # Load mask (single channel) and scale to [0, 1]
            mask_arr = img_to_array(load_img(mask_path, target_size=self.image_size, color_mode='grayscale'))
            y[i,] = (mask_arr / 255.0).astype(np.float32)
        
        return X, y

# --- Create Generators ---
preprocess_input = sm.get_preprocessing(BACKBONE)

train_generator = SegDataGenerator(train_frames, train_masks, BATCH_SIZE, IMAGE_SIZE, preprocess_input)
val_generator = SegDataGenerator(val_frames, val_masks, BATCH_SIZE, IMAGE_SIZE, preprocess_input)

Total paired samples found: 100
Train samples: 80, Validation samples: 20


In [23]:
# --- Cell 4: Training Setup ---

# --- COMMON CALLBACKS ---
# Save the best model weights based on validation Dice
# checkpoint_path = f'./{model_name}_best_model.h5'
# model_checkpoint = ModelCheckpoint(checkpoint_path, monitor='val_dice_coef', mode='max', save_best_only=True, verbose=0)
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7)


def train_and_evaluate_model(model_builder_func, model_name, epochs=50):
    print(f"\n--- Starting Training for Model: {model_name} ---")
    
    # Checkpoint to save best weights
    model_checkpoint = ModelCheckpoint(f'./{model_name}_best_model.weights.h5', 
                                       monitor='val_dice_coef', 
                                       mode='max', 
                                       save_best_only=True, 
                                       verbose=1)
    
    model = model_builder_func()
    
    # Train the model
    history = model.fit(
        train_generator,
        steps_per_epoch=len(train_generator),
        epochs=epochs,
        validation_data=val_generator,
        validation_steps=len(val_generator),
        callbacks=[early_stopping, lr_scheduler, model_checkpoint],
        verbose=1 # Use 1 for progress bars in Kaggle
    )

    # Load the best weights for final evaluation
    try:
        model.load_weights(f'./{model_name}_best_model.weights.h5')
        print(f"Loaded best weights for {model_name}.")
    except:
        print(f"Could not load saved weights for {model_name}. Using last epoch.")

    # Evaluate the model
    # The metrics order is: [loss, iou_score, dice_coef, sensitivity, specificity, accuracy]
    results = model.evaluate(val_generator, steps=len(val_generator), verbose=0)
    
    metrics_map = {
        'Model': model_name,
        'Loss (Hybrid)': results[0],
        'IoU': results[1],
        'Dice Coefficient': results[2],
        'Sensitivity (Recall)': results[3],
        'Specificity': results[4],
        'Accuracy': results[5]
    }
    print(f"Evaluation Complete for {model_name}.")
    return metrics_map, model, history

# --- 4.1 CNN Model Builders (CORRECTED) ---

def build_unet():
    # Use sm.Unet instead of just Unet
    model = sm.Unet(BACKBONE, encoder_weights='imagenet', classes=N_CLASSES, activation='sigmoid', input_shape=INPUT_SHAPE)
    model.compile(optimizer=Adam(learning_rate=1e-4), loss=hybrid_loss, metrics=METRICS)
    model.name = 'Unet_ResNet34'
    return model

def build_att_unet():
    # Fix: Use sm.AttentionUnet instead of just AttentionUnet
    model = sm.AttentionUnet(BACKBONE, encoder_weights='imagenet', classes=N_CLASSES, activation='sigmoid', input_shape=INPUT_SHAPE)
    model.compile(optimizer=Adam(learning_rate=1e-4), loss=hybrid_loss, metrics=METRICS)
    model.name = 'AttUnet_ResNet34'
    return model

def build_unet_plus():
    # Use sm.UnetPlusPlus instead of just UnetPlusPlus
    model = sm.UnetPlusPlus(BACKBONE, encoder_weights='imagenet', classes=N_CLASSES, activation='sigmoid', input_shape=INPUT_SHAPE)
    model.compile(optimizer=Adam(learning_rate=1e-4), loss=hybrid_loss, metrics=METRICS)
    model.name = 'Unet++_ResNet34'
    return model

# The Transformer model builders are fine as they already use 'models.transunet_2d' etc.
# --- 4.2 Transformer Model Builders ---
def build_trans_unet():
    # Hybrid CNN-ViT Model
    model = models.transunet_2d(
        INPUT_SHAPE, 
        filter_num=[64, 128, 256, 512, 1024], 
        n_labels=N_CLASSES, 
        stack_num_up=2, 
        embed_dim=768,
        depth=12,
        input_patch_size=16,
        activation='sigmoid', 
        name='TransUNet'
    )
    model.compile(optimizer=Adam(learning_rate=1e-4), loss=hybrid_loss, metrics=METRICS)
    return model

def build_swin_unet():
    # Pure Hierarchical Transformer Encoder Model
    model = models.swin_unet_2d(
        INPUT_SHAPE, 
        filter_num=[128, 256, 512, 1024], 
        n_labels=N_CLASSES, 
        stack_num_down=2, 
        stack_num_up=2,
        patch_size=(4, 4),
        embed_dim=128,
        num_mlp=512,
        activation='sigmoid',
        name='SwinUNet'
    )
    model.compile(optimizer=Adam(learning_rate=1e-4), loss=hybrid_loss, metrics=METRICS)
    return model

In [24]:
# --- Cell 5: EXECUTE ALL MODELS AND COMPARE ---

# List of model builders and their names
model_experiments = [
    (build_unet, "U-Net_CNN"),
    (build_att_unet, "Attention_U-Net_CNN"),
    (build_unet_plus, "U-Net++_CNN"),
    (build_trans_unet, "TransUNet_Hybrid"),
    (build_swin_unet, "Swin-UNet_Transformer")
]

all_results = []
trained_models = {}

for builder, name in model_experiments:
    results, model, history = train_and_evaluate_model(builder, name, epochs=50) # Set epochs lower for initial testing, e.g., 20
    all_results.append(results)
    trained_models[name] = model # Store models if needed for visualization

# Convert all results to a DataFrame
df_results = pd.DataFrame(all_results)
df_results.set_index('Model', inplace=True)

# --- Final Output ---
print("\n" + "="*80)
print("             🚀 FINAL SEGMENTATION MODEL COMPARISON (ResNet34 Backbone) 🚀")
print("="*80)

# Sort by the most critical metric (Dice Coefficient)
df_results_sorted = df_results.sort_values(by='Dice Coefficient', ascending=False)
print(df_results_sorted.to_markdown(floatfmt=".4f"))

# Optional: Save results
# df_results_sorted.to_csv('segmentation_comparison_final_results.csv')


--- Starting Training for Model: U-Net_CNN ---
Epoch 1/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 264ms/step - accuracy: 0.3754 - dice_coef: 0.1219 - iou_score: 0.0651 - loss: 0.8522 - sensitivity: 0.6700 - specificity: 0.3574
Epoch 1: val_dice_coef improved from -inf to 0.06393, saving model to ./U-Net_CNN_best_model.weights.h5


5/5 ━━━━━━━━━━━━━━━━━━━━ 48s 2s/step - accuracy: 0.3792 - dice_coef: 0.1242 - iou_score: 0.0664 - loss: 0.8491 - sensitivity: 0.6735 - specificity: 0.3609 - val_accuracy: 0.9092 - val_dice_coef: 0.0639 - val_iou_score: 0.0330 - val_loss: 0.6965 - val_sensitivity: 0.0288 - val_specificity: 0.9501 - learning_rate: 1.0000e-04
Epoch 2/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 266ms/step - accuracy: 0.4680 - dice_coef: 0.1349 - iou_score: 0.0725 - loss: 0.7904 - sensitivity: 0.8567 - specificity: 0.4430
Epoch 2: val_dice_coef improved from 0.06393 to 0.10118, saving model to ./U-Net_CNN_best_model.weights.h5


5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 647ms/step - accuracy: 0.4706 - dice_coef: 0.1380 - iou_score: 0.0743 - loss: 0.7880 - sensitivity: 0.8574 - specificity: 0.4450 - val_accuracy: 0.7081 - val_dice_coef: 0.1012 - val_iou_score: 0.0533 - val_loss: 0.7540 - val_sensitivity: 0.2162 - val_specificity: 0.7430 - learning_rate: 1.0000e-04
Epoch 3/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 266ms/step - accuracy: 0.5651 - dice_coef: 0.1745 - iou_score: 0.0957 - loss: 0.7390 - sensitivity: 0.8817 - specificity: 0.5381
Epoch 3: val_dice_coef improved from 0.10118 to 0.10544, saving model to ./U-Net_CNN_best_model.weights.h5


5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 645ms/step - accuracy: 0.5678 - dice_coef: 0.1722 - iou_score: 0.0943 - loss: 0.7392 - sensitivity: 0.8801 - specificity: 0.5416 - val_accuracy: 0.5336 - val_dice_coef: 0.1054 - val_iou_score: 0.0557 - val_loss: 0.7879 - val_sensitivity: 0.4114 - val_specificity: 0.5444 - learning_rate: 1.0000e-04
Epoch 4/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 272ms/step - accuracy: 0.6584 - dice_coef: 0.1646 - iou_score: 0.0898 - loss: 0.7162 - sensitivity: 0.8831 - specificity: 0.6423
Epoch 4: val_dice_coef did not improve from 0.10544
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 343ms/step - accuracy: 0.6601 - dice_coef: 0.1657 - iou_score: 0.0905 - loss: 0.7152 - sensitivity: 0.8845 - specificity: 0.6438 - val_accuracy: 0.2617 - val_dice_coef: 0.0955 - val_iou_score: 0.0502 - val_loss: 0.8591 - val_sensitivity: 0.7958 - val_specificity: 0.2339 - learning_rate: 1.0000e-04
Epoch 5/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 304ms/step - accuracy: 0.7226 - dice_coef: 0.1698 - iou_score: 0.0933 - loss: 0.6931 - 

5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 675ms/step - accuracy: 0.7242 - dice_coef: 0.1717 - iou_score: 0.0944 - loss: 0.6915 - sensitivity: 0.8684 - specificity: 0.7144 - val_accuracy: 0.2345 - val_dice_coef: 0.1136 - val_iou_score: 0.0602 - val_loss: 0.9229 - val_sensitivity: 0.8591 - val_specificity: 0.1957 - learning_rate: 1.0000e-04
Epoch 6/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 275ms/step - accuracy: 0.7758 - dice_coef: 0.1974 - iou_score: 0.1098 - loss: 0.6599 - sensitivity: 0.8966 - specificity: 0.7672
Epoch 6: val_dice_coef did not improve from 0.11364
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 353ms/step - accuracy: 0.7765 - dice_coef: 0.1969 - iou_score: 0.1095 - loss: 0.6601 - sensitivity: 0.8955 - specificity: 0.7681 - val_accuracy: 0.2790 - val_dice_coef: 0.0964 - val_iou_score: 0.0506 - val_loss: 0.9951 - val_sensitivity: 0.7934 - val_specificity: 0.2530 - learning_rate: 1.0000e-04
Epoch 7/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 278ms/step - accuracy: 0.8168 - dice_coef: 0.2223 - iou_score: 0.1253 - loss: 0.6315 - 

5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 657ms/step - accuracy: 0.8353 - dice_coef: 0.2198 - iou_score: 0.1236 - loss: 0.6256 - sensitivity: 0.8930 - specificity: 0.8321 - val_accuracy: 0.3570 - val_dice_coef: 0.1142 - val_iou_score: 0.0606 - val_loss: 0.8917 - val_sensitivity: 0.6463 - val_specificity: 0.3399 - learning_rate: 5.0000e-05
Epoch 9/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 281ms/step - accuracy: 0.8456 - dice_coef: 0.2277 - iou_score: 0.1285 - loss: 0.6172 - sensitivity: 0.8893 - specificity: 0.8440
Epoch 9: val_dice_coef did not improve from 0.11424
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 359ms/step - accuracy: 0.8460 - dice_coef: 0.2274 - iou_score: 0.1283 - loss: 0.6170 - sensitivity: 0.8909 - specificity: 0.8443 - val_accuracy: 0.3973 - val_dice_coef: 0.1056 - val_iou_score: 0.0558 - val_loss: 0.8374 - val_sensitivity: 0.4799 - val_specificity: 0.3954 - learning_rate: 5.0000e-05
Epoch 10/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 281ms/step - accuracy: 0.8522 - dice_coef: 0.2405 - iou_score: 0.1371 - loss: 0.6068 -

AttributeError: module 'segmentation_models' has no attribute 'AttentionUnet'